<a href="https://colab.research.google.com/github/qothrunnadaqobusahbillah/numeric_computation_2026/blob/main/SPL_Gauss_3_Variabel_Full_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penyelesaian Sistem Persamaan Linear 3 Variabel
## Eliminasi Gauss dengan Pivoting Parsial dan Substitusi Mundur

**Mata Kuliah:** Komputasi Numerik  
**Nama:** Qothrun Nada Qobusah Billah
**NIM:** 202511420068  
**Program Studi:** Teknik Informatika, Universitas Dr. Soetomo

In [ ]:
import numpy as np                                    # library untuk array/matriks dan operasi numerik

np.set_printoptions(precision=4, suppress=True)       # tampilkan 4 desimal; angka super kecil (mis. 1e-17) ditulis 0

In [ ]:
def gauss_spl(A, b, tampil=True):                     # fungsi penyelesaian SPL; tampil=True menampilkan proses tiap tahap
    A = np.array(A, dtype=float)                      # ubah A menjadi array desimal (hindari pembulatan bilangan bulat)
    b = np.array(b, dtype=float)                      # ubah b menjadi array desimal
    n = len(b)                                        # jumlah persamaan/variabel (di sini 3)

    M = np.hstack((A, b.reshape(-1, 1)))              # gabungkan A dan b menjadi matriks augmented [A | b] ukuran 3x4
    if tampil:                                        # cetak hanya jika tampil=True
        print("Matriks augmented awal [A | b]:")      # judul tampilan
        print(M, "\n")                                # cetak matriks awal lalu baris kosong

    # ===== TAHAP 1: ELIMINASI MAJU =====
    for k in range(n - 1):                            # k = indeks baris/kolom pivot (0 dan 1)
        p = k + np.argmax(np.abs(M[k:, k]))           # cari baris dengan |nilai| terbesar di kolom k (mulai baris k)

        if abs(M[p, k]) < 1e-12:                      # jika pivot terbesar hampir nol -> sistem singular
            raise ValueError(f"Sistem singular: pivot kolom {k+1} bernilai nol, tidak ada solusi tunggal.")  # hentikan + pesan

        if p != k:                                    # jika baris pivot bukan baris k
            M[[k, p]] = M[[p, k]]                     # tukar baris k dengan baris p (pivoting parsial)
            if tampil:                                # cetak info penukaran
                print(f"Tahap {k+1}: tukar baris {k+1} dengan baris {p+1} (pivoting parsial)")

        for i in range(k + 1, n):                     # untuk setiap baris di bawah pivot
            faktor = M[i, k] / M[k, k]                # faktor pengali = elemen yang akan dinolkan / pivot
            M[i, k:] = M[i, k:] - faktor * M[k, k:]   # Baris_i = Baris_i - faktor * Baris_k (dari kolom k ke kanan)
            M[i, k] = 0.0                             # paksa tepat 0 (hindari sisa galat pembulatan)

        if tampil:                                    # cetak hasil tahap eliminasi ini
            print(f"Matriks setelah tahap eliminasi {k+1}:")
            print(M, "\n")

    if abs(M[n - 1, n - 1]) < 1e-12:                  # cek pivot terakhir (koefisien z pada baris ke-3)
        raise ValueError("Sistem singular: pivot terakhir bernilai nol, tidak ada solusi tunggal.")  # hentikan + pesan

    # ===== TAHAP 2: SUBSTITUSI MUNDUR =====
    x = np.zeros(n)                                   # siapkan vektor solusi berisi nol
    for i in range(n - 1, -1, -1):                    # urutan i = 2, 1, 0 -> hitung z, lalu y, lalu x
        x[i] = (M[i, -1] - M[i, i + 1:n] @ x[i + 1:n]) / M[i, i]  # (konstanta - sum koef*variabel diketahui) / koef diagonal

    return x                                          # kembalikan solusi [x, y, z]

In [ ]:
def uji_kasus(nama, A, b):                            # fungsi bantu: jalankan metode lalu verifikasi hasilnya
    print("=" * 60)                                   # garis pemisah
    print(nama)                                       # judul kasus
    print("=" * 60)                                   # garis pemisah
    A = np.array(A, dtype=float)                      # pastikan A bertipe desimal
    b = np.array(b, dtype=float)                      # pastikan b bertipe desimal

    try:                                              # coba selesaikan; tangkap error jika singular
        x = gauss_spl(A, b)                           # solusi dari metode Eliminasi Gauss kita
    except ValueError as e:                           # jika gauss_spl melempar ValueError
        print("GALAT:", e)                            # tampilkan pesan error yang jelas
        print("cond(A) : {:.4e}".format(np.linalg.cond(A)))  # tampilkan bilangan kondisi (sangat besar)
        return None                                   # tidak ada solusi untuk dikembalikan

    residual = A @ x - b                              # residual = A.x - b (seharusnya mendekati 0)
    print("Solusi Gauss            : x = {:.6f}, y = {:.6f}, z = {:.6f}".format(*x))  # solusi dari metode kita
    print("Solusi np.linalg.solve  :", np.linalg.solve(A, b), "(hanya verifikasi)")   # pembanding dari NumPy
    print("Residual A.x - b        :", np.array2string(residual, formatter={'float_kind': lambda v: f'{v:.2e}'}))  # residual notasi ilmiah
    print("cond(A)                 : {:.4e}".format(np.linalg.cond(A)))  # bilangan kondisi matriks A
    return x                                          # kembalikan solusi

In [ ]:
A1 = [[ 2,  1, -1],                                   # persamaan 1:  2x +  y -  z
      [-3, -1,  2],                                   # persamaan 2: -3x -  y + 2z
      [-2,  1,  2]]                                   # persamaan 3: -2x +  y + 2z
b1 = [8, -11, -3]                                     # ruas kanan ketiga persamaan
x1 = uji_kasus("KASUS 1: Normal (solusi seharusnya 2, 3, -1)", A1, b1)  # jalankan dan verifikasi

In [ ]:
A2 = [[0, 2,  1],                                     # a11 = 0 -> tanpa pivoting terjadi pembagian dengan nol
      [1, 1,  1],
      [2, 1, -1]]
b2 = [3, 3, 2]                                        # dibuat dari solusi x = y = z = 1
x2 = uji_kasus("KASUS 2: Butuh pivoting (solusi seharusnya 1, 1, 1)", A2, b2)  # jalankan dan verifikasi

In [ ]:
A3 = [[1, 2, 3],                                      # baris 2 = 2 x baris 1 -> persamaan bergantung linear
      [2, 4, 6],
      [1, 1, 1]]
b3 = [6, 13, 3]                                       # b sengaja tidak konsisten dengan baris 1 dan 2
x3 = uji_kasus("KASUS 3: Singular (tidak punya solusi tunggal)", A3, b3)  # harus menampilkan pesan GALAT

In [ ]:
A4 = np.array([[1, 1,      1],                        # matriks hampir singular: det(A) = 1e-8
               [1, 1.0001, 1],
               [1, 1,      1.0001]])
x_benar = np.array([1.0, 2.0, 3.0])                   # solusi yang kita tetapkan sendiri
b4 = A4 @ x_benar                                     # hitung b dari solusi tersebut
x4 = uji_kasus("KASUS 4: Hampir singular / ill-conditioned (solusi seharusnya 1, 2, 3)", A4, b4)  # jalankan dan verifikasi

b4_geser = b4 + np.array([0.0, 1e-4, 0.0])            # geser SATU nilai b sebesar 0,0001 (galat relatif ~1e-5)
x4_geser = gauss_spl(A4, b4_geser, tampil=False)      # selesaikan lagi tanpa menampilkan proses
print("\nSolusi setelah b digeser 0,0001:", x4_geser)  # solusi baru
print("Perubahan solusi               :", x4_geser - x4)  # selisih terhadap solusi sebelumnya

In [ ]:
kasus = [("1. Normal",           A1, b1, x1),         # daftar kasus: (nama, A, b, solusi)
         ("2. Butuh pivoting",   A2, b2, x2),
         ("3. Singular",         A3, b3, x3),
         ("4. Ill-conditioned",  A4, b4, x4)]

print(f"{'Kasus':<22}{'Status':<12}{'Maks |residual|':<18}{'cond(A)':<12}")  # judul kolom tabel
for nama, A, b, x in kasus:                           # telusuri setiap kasus
    A_ = np.array(A, dtype=float)                     # A sebagai array desimal
    cond = np.linalg.cond(A_)                         # bilangan kondisi
    if x is None:                                     # kasus gagal (singular)
        print(f"{nama:<22}{'Singular':<12}{'-':<18}{cond:<12.2e}")
    else:                                             # kasus berhasil
        res = np.max(np.abs(A_ @ x - np.array(b, dtype=float)))  # residual terbesar
        print(f"{nama:<22}{'Berhasil':<12}{res:<18.2e}{cond:<12.2e}")

In [ ]:
A_saya = [[1,  1,  1],                                # contoh: x + y + z = 6
          [2, -1,  1],                                #         2x - y + z = 3
          [1,  2, -1]]                                #         x + 2y - z = 2
b_saya = [6, 3, 2]                                    # ubah angka-angka ini sesuai soal Anda
x_saya = uji_kasus("SISTEM SAYA (contoh solusi 1, 2, 3)", A_saya, b_saya)  # jalankan dan verifikasi